### Understanding Wrapper style for model

In [7]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import SystemMessage, ToolMessage
from pydantic import BaseModel
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, wrap_tool_call, ToolCallRequest, dynamic_prompt


#Handlet tool error handling




@wrap_tool_call
def t1(req: ToolCallRequest, handler)-> ToolMessage:

    try: 
        print("in try block")
        handler(req)
        res = ToolMessage(content=f"The tool message generated by Akhil as Dummy, The city weather curently is Awesome", tool_call_id = req.runtime.tool_call_id)
        print(res)       
        return res
    except Exception as e:
        return ToolMessage(content=f"The tool generated the Error {e}, so please check youtr input", tool_call_id = req.runtime.tool_call_id)



@tool
def get_weather_city(city: str) -> str:
    """Get the current weather for a city
    
    Args:
        city: The name of the city for which to get weather information
    
    Returns:
        A string describing the weather in the specified city
    """
    return f"The weather in {city} is sunny."

#model
basic_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '5172773b797243f6939e3f34642fbe4a.iahgs1TfHbkk9PBtNE4wY5a2'}
        }
    ) 

advanced_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3.5:397b-cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '5172773b797243f6939e3f34642fbe4a.iahgs1TfHbkk9PBtNE4wY5a2'}
        }
    ) 

# #Choose Model dynamically
# @wrap_model_call
# def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
#     print("dynamic_model_selection middleware invoked!")
#     msg:str = request.messages[0].content
#     dynamic_model = request.model
#     if  msg.find("simple")< 0:
#          print("in if")
#          dynamic_model = advanced_model
#     else: 
#         print("in else")
#         dynamic_model = request.model

#     res = handler(request.override(model=dynamic_model))    
#     print("dynamic_model_selection middleware Done!")
#     print(res)
#     return res

# @wrap_model_call
# def dynamic_tool_selection(request: ModelRequest, handler) -> ModelResponse:    
#     print("dynamic_tool_selection middleware invoked!")
#     tool_list = request.tools
#     msg:str = request.messages[0].content
#     dynamic_model = request.model
#     if  msg.find("weather")< 0:
#          print("in if")
#          tool_list = []
#     else: 
#         print("in else")
#         tool_list = [get_weather_city]

#     res = handler(request.override(tools=tool_list))
#     print("dynamic_tool_selection middleware Done!")
#     print(res)
#     return res


tool_list = [get_weather_city]


@dynamic_prompt
def d1(req: ModelRequest) -> str:
    if (req.messages[0].content.find("test") > 0):
        return "Act as a Software tester & provide your answer without using techinal jargons"
    else: 
        return req.system_message.content



@wrap_model_call
def m1(req:ModelRequest, handler)-> ModelResponse:
    print(req.system_message.content)
    return handler(req)


agent = create_agent(
    name= "PodTest Agent",
    system_prompt = "Act as a Developer & provide your answer in techinal jargons",
    model=basic_model,
    tools=tool_list,
    middleware=[ d1, m1]   
)





prompt = PromptTemplate.from_template("What is Software test case?")
promptValue = prompt.invoke({})

respo = agent.invoke({"messages": [str(promptValue)]})
print(respo)






Act as a Software tester & provide your answer without using techinal jargons
{'messages': [HumanMessage(content="text='What is Software test case?'", additional_kwargs={}, response_metadata={}, id='641654e1-6f81-48ea-bef7-5fc659bcbc56'), AIMessage(content='A software test case is a set of conditions or steps that are carried out to check if a feature or functionality of a software application works as expected. It acts like a recipe: it includes what to test, how to test it, and what result you expect. Test cases help ensure the software behaves correctly and catches any issues before it’s released.', additional_kwargs={}, response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-04-15T14:39:44.402317198Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1170285797, 'load_duration': None, 'prompt_eval_count': 315, 'prompt_eval_duration': None, 'eval_count': 71, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ol